# Topic 17 — Silhouette Analysis

> **Goal of this notebook:** learn the most useful way to **evaluate clustering quality and choose the number of clusters** without labels — the silhouette coefficient — including the per-sample silhouette plot and a from-scratch computation.

**Contents**
1. Concept & intuition
2. The mathematics (a, b, and the silhouette coefficient)
3. Using silhouette to choose k
4. The data: synthetic blobs
5. Average silhouette vs k
6. The silhouette plot (per-sample)
7. Silhouette from scratch
8. Pros, cons & when to use
9. Summary

## 1. Concept & Intuition

Clustering algorithms always return *some* clustering — but is it any good, and did we pick the right number of clusters? Without labels we need an **internal** quality measure.

The **silhouette coefficient** asks, for each point: *is it much closer to its own cluster than to the nearest other cluster?* A high score means tight, well-separated clusters. Averaged over all points it gives a single number to compare different values of `k` or different algorithms.

## 2. The Mathematics

For each point $i$:
- $a(i)$ = mean distance to **all other points in its own cluster** (cohesion).
- $b(i)$ = mean distance to points in the **nearest other cluster** (separation).

The silhouette coefficient of point $i$:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))}$$

- $s(i) \approx +1$ → well inside its cluster, far from others (great).
- $s(i) \approx 0$ → on the boundary between two clusters.
- $s(i) < 0$ → probably assigned to the **wrong** cluster.

The overall **silhouette score** is the mean of $s(i)$ over all points (range −1 to +1; higher is better).

## 3. Using Silhouette to Choose k

Unlike the elbow method (which is eyeballed), silhouette gives a number you can **maximise**: try several `k`, compute the average silhouette for each, and pick the `k` with the highest score. The per-sample **silhouette plot** adds detail — it reveals clusters that are too wide, too small, or contain misassigned points.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Data: Synthetic Blobs

In [ ]:
X, _ = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=42)
plt.scatter(X[:, 0], X[:, 1], s=12, color='gray')
plt.title('Raw data (4 true blobs)'); plt.show()

## 5. Average Silhouette vs k

In [ ]:
ks = range(2, 9)
scores = []
for k in ks:
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    scores.append(silhouette_score(X, labels))

best_k = list(ks)[int(np.argmax(scores))]
plt.plot(list(ks), scores, marker='o')
plt.axvline(best_k, color='r', ls='--', label=f'best k = {best_k}')
plt.xlabel('k'); plt.ylabel('average silhouette score')
plt.title('Silhouette score picks the best k'); plt.legend(); plt.show()
print('Best k by silhouette:', best_k, '(score=%.3f)' % max(scores))

## 6. The Silhouette Plot (per-sample)

Each bar is one point's $s(i)$, grouped by cluster. Wide, uniformly tall blocks above the average line (red) indicate healthy clusters.

In [ ]:
labels = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit_predict(X)
sample_sil = silhouette_samples(X, labels)
avg = silhouette_score(X, labels)

fig, ax = plt.subplots(figsize=(8, 6))
y_lower = 10
for i in range(best_k):
    vals = np.sort(sample_sil[labels == i])
    size = len(vals)
    color = cm.nipy_spectral(float(i) / best_k)
    ax.fill_betweenx(np.arange(y_lower, y_lower + size), 0, vals,
                     facecolor=color, edgecolor=color, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size, str(i))
    y_lower += size + 10
ax.axvline(avg, color='red', ls='--', label=f'average = {avg:.2f}')
ax.set_xlabel('silhouette coefficient s(i)'); ax.set_ylabel('cluster')
ax.set_title(f'Silhouette plot (k={best_k})'); ax.legend(); plt.show()

## 7. Silhouette From Scratch

Compute $a(i)$, $b(i)$ and $s(i)$ directly from the distance matrix and compare to scikit-learn.

In [ ]:
from scipy.spatial.distance import cdist

def silhouette_scratch(X, labels):
    D = cdist(X, X)
    clusters = np.unique(labels)
    s = np.zeros(len(X))
    for i in range(len(X)):
        own = labels[i]
        same = (labels == own)
        same[i] = False
        a = D[i, same].mean() if same.sum() > 0 else 0
        b = min(D[i, labels == c].mean() for c in clusters if c != own)
        s[i] = (b - a) / max(a, b) if max(a, b) > 0 else 0
    return s.mean()

print('From-scratch silhouette:', round(silhouette_scratch(X, labels), 4))
print('scikit-learn silhouette:', round(silhouette_score(X, labels), 4))

## 8. Pros, Cons & When to Use

**Pros**
- A single, **label-free** number to compare clusterings and choose `k`.
- The per-sample plot diagnoses *which* clusters are weak.
- Works with any clustering algorithm and distance metric.

**Cons**
- **O(n²)** distance computations → expensive on large datasets (sample it).
- Biased toward convex, globular clusters (like K-Means) — can undervalue good DBSCAN results on odd shapes.
- A single score can hide per-cluster problems (always look at the plot too).

**When to use**
- Selecting the number of clusters and validating clustering quality, especially with K-Means/hierarchical.

## 9. Summary

- The silhouette coefficient compares each point's **cohesion** $a(i)$ to its **separation** $b(i)$: $s = (b-a)/\max(a,b)$.
- Averaging gives a label-free score to **maximise over k**; here it correctly identified k=4.
- The **silhouette plot** shows per-cluster health, not just the average.
- Our from-scratch computation matched scikit-learn, confirming the formula.